In [2]:
import pandas as pd
import numpy as np

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import pickle
import warnings
warnings.filterwarnings('ignore')

In [4]:
from google.colab import files
uploaded = files.upload()

Saving dosage_opt.csv to dosage_opt.csv


In [15]:
dosage = pd.read_csv('dosage_opt.csv')
print(f"Dosage records: {len(dosage)}")
print("Sample data:")
print(dosage[['medicine_name', 'adult_dosage']].head(10))
print("\nDosage patterns:")
print(dosage['adult_dosage'].value_counts().head(10))


Dosage records: 240
Sample data:
                    medicine_name                adult_dosage
0                        abacavir                      300 mg
1                     abiraterone              250 mg; 500 mg
2             acamprosate calcium                      333 mg
3                   acetazolamide                      250 mg
4  all-trans retinoid acid (ATRA)                       10 mg
5                       alteplase         10 mg; 20 mg; 50 mg
6                   amidotrizoate  140 mg to 420 mg iodine/mL
7                       amiloride                        5 mg
8                   amitriptyline         10 mg; 25 mg; 75 mg
9                     amodiaquine            153 mg or 200 mg

Dosage patterns:
adult_dosage
50 mg       9
250 mg      8
500 mg      6
10 mg       5
10 mg/mL    4
200 mg      4
2%          4
5%          4
1 g         4
5 mg        3
Name: count, dtype: int64


In [9]:
# Prepare features
dosage['text_features'] = dosage['medicine_name'] + ' ' + dosage['adult_dosage'].fillna('')

In [10]:
# TF-IDF
tfidf_dosage = TfidfVectorizer(max_features=200, stop_words='english')
X_text_dosage = tfidf_dosage.fit_transform(dosage['text_features'])

In [19]:
def extract_dosage(dosage_text):
    if pd.isna(dosage_text):
        return None

    dosage_text = str(dosage_text).lower()

    # Extract numbers with units
    import re

    # Pattern 1: "500mg" -> 500
    mg_match = re.search(r'(\d+(?:\.\d+)?)\s*mg', dosage_text)
    if mg_match:
        return float(mg_match.group(1))

    # Pattern 2: "5ml" -> 5
    ml_match = re.search(r'(\d+(?:\.\d+)?)\s*ml', dosage_text)
    if ml_match:
        return float(ml_match.group(1))

    # Pattern 3: "250mg/5ml" -> 250
    ratio_match = re.search(r'(\d+(?:\.\d+)?)\s*mg/\d+', dosage_text)
    if ratio_match:
        return float(ratio_match.group(1))

    # Pattern 4: Just numbers "500" -> 500
    num_match = re.search(r'(\d+(?:\.\d+)?)', dosage_text)
    if num_match:
        return float(num_match.group(1))

    return None

dosage['dosage_numeric'] = dosage['adult_dosage'].apply(extract_dosage)
print(f"Extracted dosages: {dosage['dosage_numeric'].notna().sum()}")
print("Sample extracted dosages:")
print(dosage[['medicine_name', 'adult_dosage', 'dosage_numeric']].head(10))
y_dosage = dosage['dosage_numeric']

# Prepare features
dosage['text_features'] = dosage['medicine_name'] + ' ' + dosage['adult_dosage'].fillna('')

# TF-IDF
tfidf_dosage = TfidfVectorizer(max_features=200, stop_words='english')
X_text_dosage = tfidf_dosage.fit_transform(dosage['text_features'])

Extracted dosages: 197
Sample extracted dosages:
                    medicine_name                adult_dosage  dosage_numeric
0                        abacavir                      300 mg           300.0
1                     abiraterone              250 mg; 500 mg           250.0
2             acamprosate calcium                      333 mg           333.0
3                   acetazolamide                      250 mg           250.0
4  all-trans retinoid acid (ATRA)                       10 mg            10.0
5                       alteplase         10 mg; 20 mg; 50 mg            10.0
6                   amidotrizoate  140 mg to 420 mg iodine/mL           140.0
7                       amiloride                        5 mg             5.0
8                   amitriptyline         10 mg; 25 mg; 75 mg            10.0
9                     amodiaquine            153 mg or 200 mg           153.0


In [23]:
# Filter out rows with NaN in y_dosage
valid_indices = y_dosage.dropna().index
X_text_dosage_filtered = X_text_dosage[valid_indices]
y_dosage_filtered = y_dosage.loc[valid_indices]

# Train test split
X_train_dosage, X_test_dosage, y_train_dosage, y_test_dosage = train_test_split(X_text_dosage_filtered, y_dosage_filtered, test_size=0.2, random_state=42)

# Train model
rf_dosage = RandomForestRegressor(n_estimators=100, random_state=42)
rf_dosage.fit(X_train_dosage, y_train_dosage)

RandomForestRegressor(random_state=42)

In [24]:
# Evaluate
y_pred_dosage = rf_dosage.predict(X_test_dosage)
print(f"R2 Score: {r2_score(y_test_dosage, y_pred_dosage):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_dosage, y_pred_dosage)):.3f}")


R2 Score: 0.399
RMSE: 149.169


In [25]:
# Save model
pickle.dump(rf_dosage, open('dosage_model.pkl', 'wb'))
pickle.dump(tfidf_dosage, open('dosage_tfidf.pkl', 'wb'))
print("Model saved!")

Model saved!


In [26]:
from google.colab import files
files.download('dosage_model.pkl')
files.download('dosage_tfidf.pkl')

print("Dosage optimization model downloaded!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Dosage optimization model downloaded!
